© Copyright, 2026 G. Schaer.

SPDX-License-Identifier: GPL-3.0-only

# The Single Axis Control Moment Gyroscope Problem

## Introduction

In this project, we are designing a controller to orient a model spacecraft about its yaw axis. To do so, we will be using a ***double-axis control moment gyroscope (CMG)*** comprised of a rolling outer ***gimbal***, a pitching inner gimbal, and a rapidly rotating ***flywheel***. The goal of the project is to determine how to rotate both gimbals so that the spacecraft points in a desired direction. To solve this problem, we are taking an approach called "model-based control". This means that our controller design will be based on a model of the dynamics of the system. The first step, then, is to use the Lagrangian mechanics formulation to derive the equations of motion. This project is similar to the last project in this series, but we have added an additional gimbal. Doing so extends the controllability of the system and allows us to orient objects without relying on a gravity force, like in the single-axis CMG project.

## The Process

To design a model-based controller, we will take 6 steps:
1. Determine the equations of motion of the system
2. Place the equations of motion in standard form
3. Linearize the standard form equations
4. Select a gain matrix that stablizes the system
5. Build a controller with our gain matrix
6. Simulate the system

## 1. System Dynamics and Equations of Motion

Let's derive a dynamic model of the opposeed CMG system. From the diagram above, we see there are five major components, the spacecraft, called $S$, the left gimbal, called $A$, the right gimbal, called $B$, and the left flywheel, called $F_a$, and the right flywheel, called $F_b$. There are four ***generalized coordinates*** for our system: the roll of the spacecraft, $\phi$, the pitch of the spacecraft, $\theta$, the yaw of the spacecraft, $\psi$, and the gimbal angle, $\gamma$. Additionally, a torque, $\tau_\gamma$, is applied to the both gimbals.

The first step in the Lagrangian mechanics approach to deriving the equations of motion of a system is to calculate the kinetic and potential energies of the system with respect to the generalized coordinates and their derivatives. We can do this using the Python package Sympy.

In [ ]:
# Import everything we need from SymPy
import sympy as sym
import numpy as np
import scipy as sci
from sympy import Symbol, Matrix, Function, Derivative, N
from sympy import diff, simplify, trigsimp, sin, cos, solve, init_printing, symbols, lambdify
init_printing() # This function will make the outputs of SymPy look prettier and be easier to read

Now we are going to define the ***parameters*** of the system, that is, the system constants. All components have thier own moments of inertial: $I_S^S$ for the spacecraft, $I_A^A$ for the left gimbal, $I_B^B$ for the right gimbal, $I_Fa^A$ for the left flywheel, and $I_Fb^B$ for the right flywheel. Here, $I_j^k$ denotes the moment of inertia tensor of object $j$ in the inertial frame of object $k$. Additionally, the flywheels are spinning at a constant rotation rate, $\omega_F$.

In [ ]:
# Define all the system parameters
omega_F = 62.8 # The angular rate of the flywheel
I_SS = np.array([[0.00828, 0, 0],
                 [0, 0.00828, 0],
                 [0, 0, 0.00828]]) # The moment of inertia of the spacecraft in kg-m^2 in its own frame

I_AA = np.array([[0.000937, 0, 0],
                 [0, 0.000476, 0],
                 [0, 0, 0.000476]]) # The moment of inertia of the A gimbal in kg-m^2 in its own frame
I_BB = np.array([[0.000937, 0, 0],
                 [0, 0.000476, 0],
                 [0, 0, 0.000476]]) # The moment of inertia of the B gimbal in kg-m^2 in its own frame

I_FaA = np.array([[0.00207, 0, 0],
                  [0, 0.00376, 0],
                  [0, 0, 0.00207]]) # The moment of inertia of the A flywheel in kg-m^2 in the A gimbal's inertial frame
I_FbB = np.array([[0.00207, 0, 0],
                  [0, 0.00376, 0],
                  [0, 0, 0.00207]]) # The moment of inertia of the B flywheel in kg-m^2 in the B gimbal's inertial frame

Now we make symbols and functions. These are elements of SymPy and can be thought of as exactly the same as symbols (variables) and functions from math.

In [ ]:
# Time is a symbol (variable)
t = Symbol('t')

# The generalized coordinates and the inputs are both functions of time.
# This means that they are initialized as Functions.
phi_fn = Function('phi')     # The roll of the spacecraft
theta_fn = Function('theta') # The pitch of the spacecraft
psi_fn = Function('psi')     # The yaw of the spacecraft
gamma_fn = Function('gamma') # The angle of the gimbals
tau_gamma_fn = Function('tau_gamma') # The torque applied to the gimbals

Next up, we can start the process of defining the dynamics of the system. We begin by defining the set of rotation matrices that transform one inertial from to another inertial frame. After calculated, in the pursuit of calculating rotational energy, we use these transformation matrices to convert angular rates to angular velocities.

In [ ]:
# Define the standard rotation matrices
R_x = lambda x : Matrix([[1, 0,       0     ],
                         [0, cos(x), -sin(x)],
                         [0, sin(x),  cos(x)]])
R_y = lambda x : Matrix([[ cos(x), 0, sin(x)],
                         [ 0,      1, 0     ],
                         [-sin(x), 0, cos(x)]])
R_z = lambda x : Matrix([[cos(x), -sin(x), 0],
                         [sin(x),  cos(x), 0],
                         [0,       0,      1]])

# Calculate the rotation matrices between the adjacent inertial frames
R_AS = R_z(gamma_fn(t))
R_BS = R_z(-gamma_fn(t))

In [ ]:
# Get the angular velocities of each component in their own inertial frames
ang_vel_S_S = Matrix([diff(phi_fn(t), t), -diff(theta_fn(t), t), diff(psi_fn(t), t)])
ang_vel_A_A = Matrix([0, 0, diff(gamma_fn(t), t)]) + R_AS.T @ ang_vel_S_S
ang_vel_Fa_A = Matrix([0, omega_F, 0]) + ang_vel_A_A
ang_vel_B_B = Matrix([0, 0, -diff(gamma_fn(t), t)]) + R_BS.T @ ang_vel_S_S
ang_vel_Fb_B = Matrix([0, omega_F, 0]) + ang_vel_B_B

With the angular rates, we can calculate the rotational energies of the system. As well as the ***Langrangian***, which, in this case, is defined as the sum of the rotational energies (because there are not potential energies in this system).

In [ ]:
# Get the rotational energies of each component
T_S = 0.5 * (ang_vel_S_S.T @ I_SS @ ang_vel_S_S)[0, 0]
T_A = 0.5 * (ang_vel_A_A.T @ I_AA @ ang_vel_A_A)[0, 0]
T_Fa = 0.5 * (ang_vel_Fa_A.T @ I_FaA @ ang_vel_Fa_A)[0, 0]
T_B = 0.5 * (ang_vel_B_B.T @ I_BB @ ang_vel_B_B)[0, 0]
T_Fb = 0.5 * (ang_vel_Fb_B.T @ I_FbB @ ang_vel_Fb_B)[0, 0]

In [ ]:
# Calculate the Lagrangian
L = simplify(trigsimp(T_S + T_A + T_Fa + T_B + T_Fb))
N(L,3)

Given the Lagrangian, the equations of motion of the system are defined via ***Euler-Lagrange equations*** with generalized forces:
$$\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{q_j}} \right) - \frac{\partial L}{\partial q_j} - Q_j = 0$$

Where $q_j$ is the $j$th generalized coordinate of the system and $Q_j$ are the non-conservative generalized forces acting on the $j$th generalized coordinate. For this system, then, the Euler-Lagrange equations take the form:

\begin{align}
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\phi}} \right) - \frac{\partial L}{\partial \phi}&=0\\
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\theta}} \right) - \frac{\partial L}{\partial \theta}&=0\\
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\psi}} \right) - \frac{\partial L}{\partial \psi}&=0\\
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{\gamma}} \right) - \frac{\partial L}{\partial \gamma} - \tau_\gamma&=0
\end{align}

Solving these equations...

In [ ]:
# Calculate the Euler-Lagrange equations
f_phi = diff(diff(L, Derivative(phi_fn(t), t)), t) - diff(L, phi_fn(t))
f_theta = diff(diff(L, Derivative(theta_fn(t), t)), t) - diff(L, theta_fn(t))
f_psi = diff(diff(L, Derivative(psi_fn(t), t)), t) - diff(L, psi_fn(t))
f_gamma = diff(diff(L, Derivative(gamma_fn(t), t)), t) - diff(L, gamma_fn(t)) - tau_gamma_fn(t)

In [ ]:
# Solve each equation for the second order derivatives
soln = solve([f_phi, f_theta, f_psi, f_gamma], 
              Derivative(phi_fn(t), (t, 2)), 
              Derivative(theta_fn(t), (t, 2)), 
              Derivative(psi_fn(t), (t, 2)), 
              Derivative(gamma_fn(t), (t, 2)),)
soln = simplify(trigsimp(soln))
f_phi = soln[Derivative(phi_fn(t), (t, 2))]
f_theta = soln[Derivative(theta_fn(t), (t, 2))]
f_psi = soln[Derivative(psi_fn(t), (t, 2))]
f_gamma = soln[Derivative(gamma_fn(t), (t, 2))]

In [ ]:
# Assemble the second order derivative equations
f = Matrix([f_phi, f_theta, f_psi, f_gamma])

# Replace the functions with symbols to make the system look prettier
(phidot, thetadot, psidot, gammadot,
 phi, theta, psi, gamma, tau_gamma) = symbols('phidot, thetadot, psidot, gammadot, phi, theta, psi, gamma, tau_gamma')
f = f.subs({Derivative(phi_fn(t), t) : phidot,
            Derivative(theta_fn(t), t) : thetadot,
            Derivative(psi_fn(t), t) : psidot,
            Derivative(gamma_fn(t), t) : gammadot,
            phi_fn(t) : phi,
            theta_fn(t) : theta,
            psi_fn(t) : psi,
            gamma_fn(t) : gamma,
            tau_gamma_fn(t) : tau_gamma,})

# Make the solution pretty
f = simplify(trigsimp(f))

...we get a system of equations of the form:
$$
\begin{bmatrix}
\ddot{\phi} \\
\ddot{\theta} \\
\ddot{\psi} \\
\ddot{\gamma} \\
\end{bmatrix} = 
\mathbf{f}\left(\dot{\phi}, \phi, \dot{\theta}, \theta, \dot{\psi}, \psi, \dot{\gamma}, \gamma \tau_\gamma\right)
$$
where $\mathbf{f}$ is:

In [ ]:
N(f,3)

## 2. The Opposed Simplification

Notice that the numerator of the first equation of motion (relating to the roll acceleration, $\ddot{\phi}$) is multiplied by the roll rate, $\dot{\phi}$. Therefore, if the roll rate was $0$, then $\ddot{\phi}$ would also both be zero. Suppose our system starts from an initial state where the roll rate is $0$:

In [ ]:
# Assume that the initial roll and yaw rates are 0 radians / second
f = f.subs({phidot : 0.0,})
f = simplify(trigsimp(f))
N(f,3)

From this initial condition, $\ddot{\phi}$ and $\dot{\phi}$ are both $0$, therefore, we know that the roll will remain at its initial value indefinitely. Further suppose that, at our initial condition, the yaw rate, $\dot{\psi}$, is also $0$. Again, because $\ddot{\psi}$ is always $0$ regardless of initial condition, and $\dot{\psi}$ is initially $0$, the yaw will remain at its initial value indefinitely.

The stationary nature of the roll and yaw, coupled with the fact that neither $\ddot{\theta}$ nor $\ddot{\gamma}$ are dependent on the roll or yaw, effectively removes the roll and yaw as the degrees of freedom of the system. As such, they may be dropped from the equations of motion. This symmetry is caused by the opposed nature of the two control moment gyroscopes. As $A$ rotates counterclockwise, its flywheel's angular momentum vector rotates about the $+z$ axis, and as $B$ rotates clockwise (as enforced by the gears between them), its flywheel's angular momentum vector rotates about the $-z$ axis. This symmetric movement perfectly cancels reaction torques induced about all axes except the pitch axis.

Therefore, we end up the the simplified system:
$$
\begin{bmatrix}
\ddot{\theta} \\
\ddot{\gamma} \\
\end{bmatrix} = 
\mathbf{f}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right)
$$
where $\mathbf{f}$ is:

In [ ]:
f = Matrix([f[1], f[3]])
N(f,3)

## 2. Standard Form

For model-based controls, a ***standard form*** of equations of motion is typically used:
$$
\dot{\mathbf{m}} = f(\mathbf{m}, \mathbf{n}),
$$

where $\mathbf{m}$ is the ***nonlinear state vector*** and $\mathbf{n}$ is the ***nonlinear input vector***.

There are two major differences between our current system and a system in standard form:
1. All the equations in our system are *second order*. Standard form requires all equations to be *first order*.
2. There are dervative arguments in our current system, $f$. Standard form requires that all arguments to $f$ are *zeroth order*. 

To fix problem 1, we need to replace both our *second order* differential equations with two *first order* differential equations. To do this, we can define two new variables: $\omega_{\theta}$ and $\omega_{\gamma}$. We define these new variables via the differential equations:
\begin{align}
\dot{\theta} &= \omega_{\theta},\\
\dot{\gamma} &= \omega_{\gamma}.\\
\end{align}

Now we expand our current system by inserting these new differential equations:
$$
\begin{bmatrix}
\ddot{\theta} \\
\dot{\theta} \\
\ddot{\gamma} \\
\dot{\gamma} \\
\end{bmatrix} = 
\begin{bmatrix}
f_{\theta}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right) \\
\omega_\theta \\
f_{\gamma}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right) \\
\omega_\gamma \\
\end{bmatrix}
$$

Next, by taking a derivative of our new differential equations, we find
\begin{align}
\ddot{\theta} &= \dot{\omega_{\theta}},\\
\ddot{\gamma} &= \dot{\omega_{\gamma}}.\\
\end{align}

Therefore, we can replace the second order differential equations with first order differential equations via a change of variables:
$$
\begin{bmatrix}
\dot{\omega_\theta} \\
\dot{\theta} \\
\dot{\omega_\gamma} \\
\dot{\gamma} \\
\end{bmatrix} = 
\begin{bmatrix}
f_{\theta}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right) \\
\omega_\theta \\
f_{\gamma}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right) \\
\omega_\gamma \\
\end{bmatrix}
$$

This new system is a *first order* system of differential equations, which is what we wanted. To replicate this in code, we can do the following two steps:

In [ ]:
# Define our new variables
(omega_theta, omega_gamma) = sym.symbols('omega_theta, omega_gamma')

# Add them to the system
f = sym.Matrix([f[0], omega_theta, f[1], omega_gamma])

Now we need to fix problem 2: there are dervative arguments in our current system. Standard form requires that all arguments to the system of differential equations are *zeroth order*. The good news is that we already have all the tools we need to correct this problem. To begin, note that the derivative arguments are in the functions
\begin{align}
\mathbf{f}\left(\dot{\theta}, \theta, \dot{\gamma}, \gamma, \tau_\gamma \right)
\end{align}

Recall that we have already created the variables $\omega_{\theta}$ and $\omega_{\gamma}$ which are defined as
\begin{align}
\omega_{\theta} &= \dot{\theta},\\
\omega_{\gamma} &= \dot{\gamma}.\\
\end{align}
Therefore, to ensure all arguments to the system are *zeroth order*, we can simply replace every instance of $\dot{\theta}$ and $\dot{\gamma}$ with $\omega_{\theta}$ and $\omega_{\gamma}$, respectively. This results in a system with the form:
$$
\begin{bmatrix}
\dot{\omega_\theta} \\
\dot{\theta} \\
\dot{\omega_\gamma} \\
\dot{\gamma} \\
\end{bmatrix} = 
\begin{bmatrix}
f_{\theta}\left(\omega_\theta, \theta, \omega_\gamma, \gamma, \tau_\gamma \right) \\
\omega_\theta \\
f_{\gamma}\left(\omega_\theta, \theta, \omega_\gamma, \gamma, \tau_\gamma \right) \\
\omega_\gamma \\
\end{bmatrix}
$$

Every argument to this system is now *zeroth order*, which is what we wanted. To replicate this in code, we can do the following one step:

In [ ]:
# Substitute the derivative terms with the new variables
f = f.subs({thetadot : omega_theta, 
            gammadot : omega_gamma,})

If we define the nonlinear state vector like this:
$$
\mathbf{m} = 
\begin{bmatrix}
\dot{\omega_\theta} \\
\dot{\theta} \\
\dot{\omega_\gamma} \\
\dot{\gamma} \\
\end{bmatrix},
$$
and the nonlinear input vector like this:
$$
\mathbf{n} = \begin{bmatrix} \tau_\gamma \end{bmatrix},
$$
then we can rewrite the system we have created like this:
$$
\dot{\mathbf{m}} = f(\mathbf{m},\mathbf{n})
$$
which is standard form.

In [ ]:
# Print the standard form equations of motion
f = simplify(trigsimp(f))
N(f,3)

## 3. Linearizing the System

Now we want to approximate the nonlinear system dynamics near an equilibrium point by linearizing the system to have the form
$$
\dot{\mathbf{x}} = A\mathbf{x} + B\mathbf{u},
$$
where $\mathbf{x}$ is the ***linear state vector*** and $\mathbf{u}$ is the ***linear input vector***. We call this model ***state space form***. There are three steps we take to convert our current nonlinear, standard form system of equations into state space form:
1. Choose an ***equilibrium point*** towards which the controller will drive the system.
2. Define $\mathbf{x}$ and $\mathbf{u}$.
3. Calculate $A$, also called the ***state matrix***.
4. Calculate $B$, also called the ***input matrix***.

Let's begin by finding an equilibrium point. An equilibrium point is any combination of nonlinear state, $\mathbf{m_{e}}$, and nonlinear input, $\mathbf{n_{e}}$, such that
$$
f(\mathbf{m_{e}}, \mathbf{n_{e}}) = 0.
$$
This is also called a ***stationary point*** because the time derivative of the nonlinear state vector—also called the nonlinear velocity vector—is exactly zero. Remember, we already have $f$, so all we need to do is find a valid $\mathbf{m_{e}}$ and $\mathbf{n_{e}}$. This equilibrium point will be the state that our controller drives the system towards, i.e., if our controller works, as time goes to infinity, the system will go to this equilibrium point. This is called ***stablization***. Accordingly, it would make sense to choose the equilibrium point where no torque is being applied ($\tau_\gamma=0$). Let's check to see how this limits the range of possible equilibrium points:

In [ ]:
sub_expr = {omega_theta : 0.0, # To be stationary, all the derivative terms must be 0
            omega_gamma : 0.0, # To be stationary, all the derivative terms must be 0
            tau_gamma : 0.0, } # We want the torques to be 0 at equilibrium
equil_f = f.evalf(subs=sub_expr)
print('Equilibrium Condition:')
N(equil_f, 3)

Because the equations of motion evaluate to $\mathbf{0}$, we find that these constraints guarantee an equilibrium point. This means that, as long as the state derivatives and inputs are 0, we can select any equilibrium pitch, $\theta_e$ or equilibrium gimbal angle, $\gamma_e$, we like and know we will have found an equilibrium point, i.e.,

\begin{align}
\forall \mathbf{m}_e\in \mathbb{R}^4\quad \text{such}\ \text{that}\quad f(\mathbf{m}_e, \mathbf{0})=\mathbf{0},\\
f(\mathbf{m}_e+\begin{bmatrix}0\\\theta_e\\0\\\gamma_e\end{bmatrix}, \mathbf{0})=\mathbf{0}\quad \forall \theta_e, \gamma_e \in \mathbb{R}.
\end{align}

This is an interesting property as it will give us some freedom in our controller design later on. Continuing, let's now apply these equilibrium constraints and calculate the resultant ***state and input matrices***.

In [ ]:
# Define our equilibrium condition
theta_e, gamma_e = symbols('theta_e, gamma_e')
equil_point = {omega_theta : 0.0,
               theta: theta_e,
               omega_gamma : 0.0,
               gamma : gamma_e,
               tau_gamma : 0.0, }

Recall, in the previous project in this series we defined 
\begin{align}
\mathbf{x} &= \mathbf{m} - \mathbf{m_{e}}\\
\mathbf{u} &= \mathbf{n} - \mathbf{n_{e}}\\
A &= \left.\frac{\partial f}{\partial \mathbf{m}}\right|_{\mathbf{m_{e}}, \mathbf{n_{e}}}\\
B &= \left.\frac{\partial f}{\partial \mathbf{n}}\right|_{\mathbf{m_{e}}, \mathbf{n_{e}}}.\\
\end{align}
Doing so allows us to approximate our equations of motion $\dot{\mathbf{m}}=f(\mathbf{m},\mathbf{n})$ as
$$
\dot{\mathbf{x}} \approx A\mathbf{x} + B\mathbf{u},
$$
which is state space form. Let's calculate $A$ and $B$ now.

In [ ]:
# First we define the nonlinear state vector
m = [omega_theta, theta, omega_gamma, gamma]

# Then we calculate the Jacobian of f with respect to the nonlinear state vector
A = f.jacobian(m)

# And finally, we evaluate the Jacbobian at the selected equilibrium point
A = A.evalf(subs=equil_point)
print('A=')
N(A,3)

Interestingly, even though we can select any $\gamma_e$ and guarantee an equilibrium point, it appears that our linear state matrix is dependent on $\gamma_e$. Conversely, $A$ is fully independent of $\theta_e$. Let's go ahead an choose a specific $\gamma_e$ so that $A$ has a defined numerical value. For now, we are (somewhat arbitrarily) choosing $\gamma_e=\frac{\pi}{2}$.

In [ ]:
# Define our updated equilibrium condition where gamma_e = 0.5*np.pi
equil_point = {omega_theta : 0.0,
               theta: theta_e,
               omega_gamma : 0.0,
               gamma : 0.5*np.pi,
               tau_gamma : 0.0, }
# Recalculate A at the new equilibrium point
A = f.jacobian(m)
A = A.evalf(subs=equil_point)
print('A=')
N(A,3)

Intuitively, what we have discovered is that our $A$ has the special property that it is just as accurate an approximation of the dynamics of our nonlinear system at our specific equilibrium point, $(\mathbf{m}_e, \mathbf{n}_e)$, as it is at a different equilibrium point, $\left(\mathbf{m}_e+\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix},\ \ \mathbf{n}_e\right)$.

Additionally note that the second column of $A$ is $\mathbf{0}$. Because of this, $A$ has another special property that, for all $\theta_{des}\in\mathbb{R}$,
$A\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix}=\mathbf{0}$.

Let's keep these properties in mind for now and move on to calculate $B$.

In [ ]:
# First we define the nonlinear input vector
n = [tau_gamma]

# Then we calculate the Jacobian of f with respect to the nonlinear input vector
B = f.jacobian(n)

# And finally, we evaluate the Jacbobian at the selected equilibrium point
B = B.evalf(subs=equil_point)
print('B=')
N(B,3)

Here we can also note that $B$ is fully independent of $\theta_e$. As such, by the same logic as above, $B$ also has the special property that it is just as accurate an approximation of the dynamics of our nonlinear system at our specific equilibrium point, $(\mathbf{m}_e, \mathbf{n}_e)$, as it is at a different equilibrium point, $\left(\mathbf{m}_e+\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix},\ \ \mathbf{n}_e\right)$.

## 4. Controller Design

### 4a. Reference Tracking

In the previous section, during the linearization of our system we found that 

1. For our particular system, $(\mathbf{m}_e, \mathbf{n}_e)$ and $\left(\mathbf{m}_e+\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix},\ \ \mathbf{n}_e\right)$ are equilibrium points for all $\theta_{des}$,
2. $(A, B)$ models the dynamics of the nonlinear system near $(\mathbf{m}_e, \mathbf{n}_e)$ just as well as it models the dynamics near $\left(\mathbf{m}_e+\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix},\ \ \mathbf{n}_e\right)$, and
3. $A\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix}=\mathbf{0}$.

Let's now investigate what these properties can be used for.

Suppose we wanted to pitch our spacecraft to $\theta=0$ and later at $\theta=\frac{\pi}{12}$. Were our linear dynamics dependent on $\theta$, to do this we would need to calculate a set of control gains for the two different equilibrium points and then determine a method of switching between our controllers while maintaining system stability. However, because our system has the special properties listed above, we can instead utilize a method called ***reference tracking*** to drive our system to any $\theta_{des}$ we want without modifying the controller.

The controllers we have designed so far converge the system to a specific equilibrium point. That is
$$
\mathbf{m}(t) \rightarrow \mathbf{m}_e \quad \text{as}\quad t \rightarrow \infty.
$$
As such, $\mathbf{x}(t) = \mathbf{m}(t) - \mathbf{m}_e$ implies
$$
\mathbf{x}(t) \rightarrow 0\quad \text{as}\quad t \rightarrow \infty.
$$

Suppose instead we want the system to converge to a different equilibrium point. That is
$$
\mathbf{m}(t) \rightarrow \mathbf{m}_e+\begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix} \quad \text{as}\quad t \rightarrow \infty,\quad \theta_{des}\in\mathbb{R}.
$$
As such, $\mathbf{x}(t) = \mathbf{m}(t) - \mathbf{m}_e$ now implies
$$
\mathbf{x}(t) \rightarrow \begin{bmatrix}0\\\theta_{des}\\0\\0\end{bmatrix}=\mathbf{x}_{des}\quad \text{as}\quad t \rightarrow \infty,\quad \theta_{des}\in\mathbb{R}.
$$
Our above analysis of the system guarantees that $\mathbf{m}_{e} + \mathbf{x}_{des}$ is an equilibrium point.

As it turns out, to do this, we can define a new variable $\mathbf{z}=\mathbf{x}-\mathbf{x}_{des}$, and a new control law such that 
$$
\mathbf{u}=-K\mathbf{z}.
$$
Given these changes, let's look at the system dynamics:
\begin{align}
\dot{\mathbf{z}} &= \frac{d}{dt}(\mathbf{x} - \mathbf{x}_{des}) \\
&= \dot{\mathbf{x}} - 0 \\
&= \dot{\mathbf{x}}.
\end{align}


Because we already showed that our state space model is just as accurate near $(\mathbf{m}_e, \mathbf{n}_e)$ as it is near the equilibrium point $(\mathbf{m}_e+\mathbf{x}_{des}, \mathbf{n}_e)$, we know that, even when near $(\mathbf{m}_e+\mathbf{x}_{des}, \mathbf{n}_e)$ instead of $(\mathbf{m}_e, \mathbf{n}_e)$, we have $\dot{\mathbf{x}}\approx A\mathbf{x}+B\mathbf{u}$. Therefore
\begin{align}
\dot{\mathbf{z}} &= \dot{\mathbf{x}} \\
&\approx A\mathbf{x}+B\mathbf{u} \\
&= A(\mathbf{z}+\mathbf{x}_{des})+B\mathbf{u} \\
&= A\mathbf{z}+B\mathbf{u}+A\mathbf{x}_{des}.
\end{align}


We also already showed that $A$ has the special property that $A\mathbf{x}_{des}=0$, therefore
\begin{align}
\dot{\mathbf{z}} &= A\mathbf{z}+B\mathbf{u}+A\mathbf{x}_{des} \\
&= A\mathbf{z}+B\mathbf{u}. \\
\end{align}
By our selected control law $\mathbf{u}=-K\mathbf{z}$, we have
\begin{align}
\dot{\mathbf{z}} &= A\mathbf{z}+B\mathbf{u} \\
&= A\mathbf{z}+B(-K\mathbf{z}) \\
&= (A-BK)\mathbf{z}.
\end{align}


We recognize this ***closed loop response matrix***, $A-BK$, as the same one from the previous pendulum cart series. As such, we have already shown that this system is ***asymptotically stable***, i.e. $\mathbf{z} \rightarrow \mathbf{0}$ as $t \rightarrow \infty$, as long as the real parts of the eigenvalues of the closed loop response matrix are negative. We have also developed methods of calculating the gain matrix $K$ to ensure this condition. Therefore, we know we can build a system such that
$$
\mathbf{z}(t) \rightarrow 0\quad \text{as}\quad t \rightarrow \infty.
$$
Recalling the definition of $\mathbf{z}$, this is identical to the statements:
\begin{align}
\mathbf{x}(t) \rightarrow \mathbf{x}_{des}\quad &\text{as}\quad t \rightarrow \infty, \\
\mathbf{m}(t) \rightarrow \mathbf{m}_e+\mathbf{x}_{des} \quad &\text{as}\quad t \rightarrow \infty.
\end{align}
which is exactly what we wanted.


Summarizing, we showed that if the following conditions are met:

1. If, for some dynamic system, both $(\mathbf{m}_e, \mathbf{n}_e)$ and $(\mathbf{m}_e+\mathbf{x}_{des}, \mathbf{n}_e)$ are equilibrium points, and
2. The linearization of the dynamic system, $(A, B)$, models the dynamics of the system near $(\mathbf{m}_e, \mathbf{n}_e)$ just as well as it models the dynamics near $(\mathbf{m}_e+\mathbf{x}_{des}, \mathbf{n}_e)$, and
3. $A\mathbf{x}_{des}=0$,

then 

1. We can select a new linear state vector $\mathbf{z}=\mathbf{x}-\mathbf{x}_{des}$ whose linear dynamics are modeled by $\dot{\mathbf{z}} = A\mathbf{z}+B\mathbf{u}$, and
2. The control law $\mathbf{u}=-K\mathbf{z}$ results in the system behavior: $\mathbf{x}(t) \rightarrow \mathbf{x}_{des}\quad \text{as}\quad t \rightarrow \infty$ if and only if the real parts of the eigenvalues of $A-BK$ are all negative.


### 4b. Controllable and Uncontrollable Subspaces

Recall, our goal is to select a set of control inputs, $\mathbf{u}(t)$, that will drive the system, $(A, B)$, to the selected reference tracked state, $\mathbf{m}_e+\mathbf{x}_{des}$. In the previous project in the CMG series, we already showed that, if our system is controllable, then we can find such a set of inputs. Let's check the controllability of our state space model now:

To calculate the ***controllability matrix***, $\mathcal{W}$, of our system, $(A,B)$, we can use `numpy`'s `hstack` function:

In [ ]:
# Calculate the controllability matrix
ctrb = tuple(np.linalg.matrix_power(A,i) @ B for i in range(A.shape[0]))
ctrb = np.hstack(ctrb).astype(float)
print('Controllability Matix:')
N(Matrix(ctrb), 3)

Next, we will use `scipy.linalg`'s `svdvals` function to get the singular values, $s$, of the controllability matrix.

In [ ]:
# Get the singular value decomposition of C
s = sci.linalg.svdvals(ctrb)

Finally, to determine if the system is controllable, we will ensure that the number of nonzero singular values is exactly equal to the number of states:

In [ ]:
# Determine controllability by comparing the number of nonzero single values to the number of states
n_nonzero_singular_vals = np.sum(~np.isclose(s, 0)) # Account for numerical errors with np.isclose
is_controllable = n_nonzero_singular_vals == len(m)
print(f"Is Controllable: {is_controllable}")

Because the number of nonzero singular values of $\mathcal{W}$ is *not* equal to the number of states, our system is *not* controllable. We can confirm this by printing the singular values:

In [ ]:
print('Singular Values of the Controllability Matrix:')
N(Matrix(np.round(s, 3)),3)

Our system, $(A, B)$, may not be controllable; however, this does not mean that there is not hope to control it. Previously we proved that a set of control inputs that drive the system to the selected equilibrium state will exist if the system is controllable, but we did not prove that, if the system is uncontrollable, no such set of inputs exist. Indeed, this is not the case.

In the previous project in this series, we proved that the span of the column vectors of the controllablity matrix defines the ***controllable subspace*** for our system $(A, B)$. Because one of the singular values of the controllability matrix is zero, we know that the column vectors do not not span the entirety of state space (which in this case is $\mathbb{R}^4$ because there are four states). Therefore, we conclude that there is a subspace of the state space we cannot control. We call this subspace the ***uncontrollable subspace*** of our system, $(A, B)$.

In this case, we can control a 3D subspace of the total 4D state space and not control a 1D subspace of the total 4D state space. In terms of eigenvalues, this is equivalent to saying that we can place three of the eigenvalues of the closed-loop response matrix anywhere we want, while the other one will remain in the same location no matter what gain matrix we choose. The eigenvalues we can place are often called ***controllable modes***, while the eigenvalues we can not place are often called or ***uncontrollable modes***.

Let's determine what the controllable and uncontrollable subspaces are for our system. Again, we know that the column vectors of the controllability matrix span the controllable subspace. `scipy`'s `linalg` module provides a function called `orth` that, given an input matrix, constructs an orthonormal basis for the ***range*** of the input matrix. In this case, range is another term for the span of the column vectors of a matrix. This gives us a method with which we can construct an orthogonal basis for the controllable subspace. Let's implement this now:

In [ ]:
# Get a basis for the controllable subspace
basis_c = sci.linalg.orth(ctrb)

# Correct numerical errors
basis_c[abs(basis_c) <= 1.0e-6] = 0.0
basis_c = basis_c / np.sum(basis_c**2, axis=0)

# Make largest mag entries in a col + (just to make prettier)
basis_c = np.array([col if (col[np.argmax(abs(col))]>=0) else -col for col in basis_c.T]).T 

# Order the columns according to their alignment with the cardinal directions
alignment = [[np.dot(col,col)-2.0*col[i]+1 for col in basis_c.T] for i in range(len(basis_c))]
ordering = np.argmin(alignment, axis=0)
basis_c = np.array([col for _, col in sorted(zip(ordering, basis_c.T))]).T

# Print results
print("Controllable Subspace:")
N(Matrix(basis_c), 5)

As expected, the dimension of this space (the number of vectors in the basis set) is three, which is exactly equal to the number of controllable modes.

Next, let's look at the uncontrollable subspace of $(A, B)$. A common result from linear algebra is that, given an $m\times n$ matrix, the null space of the transpose of the matrix forms an orthogonal complement of the range of the matrix within the matrix's codomain, $\mathbb{R}^m$. The null space of the transpose of the matrix is also called the ***left null space***. What this means intuitively is that:

Given an $m\times n$ matrix, $A$,

1. If a vector is a member of the left null space of $A$, then it is not a member of the range of $A$ and vice versa (orthogonality).
2. The sum of the dimensions of the left null space and range equals the total dimension of the codomain, $m$, (complimentary).
3. Any vector in the codomain, $\mathbb{R}^m$, can be uniquely decomposed into a component in the range and a component in the left null space.

Given these facts, we conclude that the left null space of the controllability matrix is equivalent to the uncontrollable subspace of our system $(A, B)$. To find an orthogonal basis of the left null space of the controllability matrix, we can use `scipy.linalg`'s `null_space` function:

In [ ]:
# Get a basis for the uncontrollable subspace
basis_u = sci.linalg.null_space(ctrb.T)

# Correct numerical errors
basis_u[abs(basis_u) <= 1.0e-6] = 0.0
basis_u = basis_u / np.sum(basis_u**2, axis=0)

# Make largest mag entries in a col + (just to make prettier)
basis_u = np.array([col if (col[np.argmax(abs(col))]>=0) else -col for col in basis_u.T]).T 

# Order the columns according to their alignment with the cardinal directions
alignment = [[np.dot(col,col)-2.0*col[i]+1 for col in basis_u.T] for i in range(len(basis_u))]
ordering = np.argmin(alignment, axis=0)
basis_u = np.array([col for _, col in sorted(zip(ordering, basis_u.T))]).T

# Print results
print("Uncontrollable Subspace:")
N(Matrix(basis_u), 5)

This provides us a basis of the uncontrollable subspace of $(A, B)$. As expected, the dimension of this space (the number of vectors in the basis set) is one, which is exactly equal to the number of uncontrollable modes. Additionally, the dimension of the controllable subspace and the dimension of the uncontrollable subspace sum to exactly the dimension of the state space, i.e., $3+1=4$.

Looking at these spaces, we find that $\gamma$ direction (plus a tiny bit of $\omega_\theta$ direction) is uncontrollable; however, all other directions in state space are controllable. So, is it possible to design a system that stabilizes the states along the controllable directions but allows the uncontrollable directions to evolve according to thier natural dynamics? To answer this question, we must first investigate the concepts of algebraic equilvalence, realizations, and Kalman decomposition.

### 4c. Algebraic Equivalence

Our goal is to split our state space model into a ***algebraically equivalent*** model where the controllable and uncontrollable components are seperated. Two $m\times n$ matrices, $R$ and $S$, are called algebraically equivalent if there exists an invertible $n \times n$ matrix, $P$, and an invertible $m \times m$ matrix, $Q$, such that:
$$
R = Q^{-1}SP.
$$
These are called algebraically equivalent matrices because they represent the exact same linear transformation just under two different bases.

For example, let's look at the case with square matrices. We know that a vector, $\mathbf{x}$, can be defined in two different bases, say $V$ and $W$. Let $\mathbf{x}^V$ be $\mathbf{x}$ expressed in the $V$ basis and let $\mathbf{x}^W$ be $\mathbf{x}$ expressed in the $W$ basis. Let $T$ be the ***change-of-base matrix*** that takes $W$ to $V$ such that $T\mathbf{x}^W = \mathbf{x}^V$. If $R$ and $S$ are algebraically equivalent, then they must represent the exact same linear transformation. This means that $T(S\mathbf{x}^W) = R\mathbf{x}^V$. 

It follows that 
\begin{align}
TS\mathbf{x}^W &= R\mathbf{x}^V \\
TS\mathbf{x}^W &= RT\mathbf{x}^W \\
S\mathbf{x}^W &= T^{-1}RT\mathbf{x}^W \\
S &= T^{-1}RT,
\end{align}

which is where the above definition of algebraic equivalence arises (for square matrices).

### 4d. Realizations

A ***realization*** of a state-space model is an implementation of the linearized dynamics of a system's near an equilibrium point. In other words, given some (in our case reference tracked) state-space model
$$
\dot{\mathbf{z}} = A\mathbf{z}+B\mathbf{u},
$$
a realization of $(A, B)$, is any double, $(\tilde{A}, \tilde{B})$, that represent the same dynamics in a different base. In terms of algebraic equivalency, a realization $(\tilde{A}, \tilde{B})$ must be algebraically equivalent to $(A, B)$.

For example, suppose there exists some change-of-base matrix, $T$, that takes the cardinal basis of the state space to some new basis, $V$:

$$
T\tilde{\mathbf{z}} = \mathbf{z}.
$$

Given this transform, we find

\begin{align}
\frac{d}{dt}\left(T\tilde{\mathbf{z}}\right) &= \dot{\mathbf{z}}\\
T \dot{\tilde{\mathbf{z}}} &= \dot{\mathbf{z}}\\
T \dot{\tilde{\mathbf{z}}} &= A\mathbf{z}+B\mathbf{u}\\
\dot{\tilde{\mathbf{z}}} &= T^{-1}\left(A\mathbf{z}+B\mathbf{u}\right)\\
 &= T^{-1}\left(AT\tilde{\mathbf{z}}+B\mathbf{u}\right)\\
 &= T^{-1}AT\tilde{\mathbf{z}}+T^{-1}B\mathbf{u}\\
 &= T^{-1}AT\tilde{\mathbf{z}}+T^{-1}BI\mathbf{u}.
\end{align}

If we call

\begin{align}
\tilde{A} &= T^{-1}AT \\
\tilde{B} &= T^{-1}BI,
\end{align}

from our above definition of algebraic equivalence, we conclude that $\tilde{A}$ is algebraically equivalent to $A$ and that $\tilde{B}$ is algebraically equivalent to $B$. This tells us that the system $(\tilde{A}, \tilde{B})$ does indeed represent the same dynamics as $(A, B)$, just in a different base. Therefore, $(\tilde{A}, \tilde{B})$ is a realization of $(A, B)$.

In our current model, $(A,B)$, the dynamics are represented in the standard basis, i.e., they operate on vectors expressed as linear combinations of the cardinal directions in state space. From our above analysis, if we can find a change-of-base matrix, $T$, that takes the standard basis to a basis aligned with the controllable and uncontrollable directions, we would be able to find a realization that operates on vectors expressed as linear combinations of the controllable and uncontrollable directions. Doing so would seperate the dynamics into the controllable and uncontrollable components.

### 4e. Kalman Decomposition

Such a process is called ***Kalman decomposition***. A Kalman decomposition provides a mathematical means to convert a realization of a state-space model to a new realization in which the system can be decomposed into a form with seperated controllable and uncontrollable components.

Let $\mathcal{C}$ be a matrix whose columns span the controllable subspace of some system $(A, B)$. Let $\mathcal{U}$ be a matrix whose columns span the uncontrollable subspace of some system $(A, B)$. Let $T$ be a change-of-basis matrix such that $T=\begin{bmatrix}\mathcal{C}&\mathcal{U}\end{bmatrix}$. 

How can we be sure this is a valid change-of-base matrix? Recall $\mathcal{C}$ we defined to span range of the controllability matrix and $\mathcal{U}$ was defined to span the left null space of the controllability matrix. Therefore, the range and left null space of the controllability matrix are orthogonal compliments. Because the sum of their dimensions is $4$, and given that our state space is also 4D, by the properties of othogonal compliments, we conclude that any vector in state space can be uniquely decomposed into a component in $\mathcal{C}$ and a component $\mathcal{U}$. This tells us that $T$ must be a valid change-of-basis matrix.

Given this validity, we have
$$
T^{-1}\mathbf{z} = \tilde{\mathbf{z}} = \begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix},
$$
where $\mathbf{z}_\mathcal{C}$ is the projection of $\mathbf{z}$ onto the controllable subspace and $\mathbf{z}_\mathcal{U}$ is the projection of $\mathbf{z}$ onto the uncontrollable subspace. This operation seperates the state, $\mathbf{z}$, into its controllable and uncontrollable components, which is exactly what we want.

Additionally, because $T$ is a change-of-basis matrix, we can use it to generate a realization of $(A,B)$ by the procedure listed in the above section:
\begin{align}
\dot{\tilde{\mathbf{z}}} &= \tilde{A} \tilde{\mathbf{z}} + \tilde{B}\mathbf{u} \\
\tilde{A} &= \begin{bmatrix}\mathcal{C}&\mathcal{U}\end{bmatrix}^{-1}A\begin{bmatrix}\mathcal{C}&\mathcal{U}\end{bmatrix} \\
\tilde{B} &= \begin{bmatrix}\mathcal{C}&\mathcal{U}\end{bmatrix}^{-1}B.
\end{align}

We can also decompose this realization into controllable and uncontrollable blocks. To do so, all we need to do is split $\tilde{A}$ and $\tilde{B}$ into block matrices of specific sizes and split \tilde{x} into the controllable and uncontrollable elements:
\begin{align}
\dot{\tilde{\mathbf{z}}} &= \tilde{A} \tilde{\mathbf{z}} + \tilde{B}\mathbf{u} \\ 
\begin{bmatrix}\dot{\mathbf{z}_\mathcal{C}} \\ \dot{\mathbf{z}_\mathcal{U}} \end{bmatrix} &= \begin{bmatrix}A_\mathcal{C} & A_{12} \\ 0 & A_\mathcal{U}\end{bmatrix} \begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix} + \begin{bmatrix}B_\mathcal{C} \\ 0 \end{bmatrix}\mathbf{u}.
\end{align}
In this decomposition, $A_\mathcal{C}$ is $m_c\times m_c$ where $m_c$ is the number of controllable modes, $A_\mathcal{U}$ is $m_u \times m_u$ where $m_u$ is the number of uncontrollable modes, $A_{12}$ is $m_c\times m_u$, the $0$ in $\tilde{A}$ is  $m_u\times m_c$, $B_\mathcal{C}$ is $m_c\times n$ where $n$ is the number of inputs to the system, and the $0$ in $\tilde{B}$ is  $m_u\times n$.

Investigating our new realization of $(A, B)$, we find that the controllable and uncontrollable components are fully seperated, which is exactly what we want! 

Let's implement this in code:

In [ ]:
# Build a transform that projects state space onto the controllable and uncontrollable subspaces
T = np.hstack((basis_c, basis_u))

# Build the algebraically equivalent realization of (A, B) based on T
A_tilde = T.T @ np.array(A, dtype=float) @ T
B_tilde = T.T @ np.array(B, dtype=float)

# Correct numerical instabilities
A_tilde[abs(A_tilde) <= 1.0e-6] = 0.0 
B_tilde[abs(B_tilde) <= 1.0e-6] = 0.0 

# Get the controlled, uncontrolled, and input dimensionalities
m_c = basis_c.shape[1]
m_u = basis_u.shape[1]

# Decompose the new realization into controlled, uncontrolled, and coupling blocks
A_c = A_tilde[0:m_c, 0:m_c]
A_12 = A_tilde[0:m_c, m_c:m_c+m_u]
A_u = A_tilde[m_c:m_c+m_u:, m_c:m_c+m_u]
B_c = B_tilde[0:m_c, :]

Recall the goal of this entire process was to design a controller that only stabilizes controllable modes of $(A, B)$ while making no changes to the uncontrollable modes. Given our new realization of $(A, B)$, we can now do this via a process called ***partial state feedback***.

### 4f. Partial State Feedback

In previous projects, we have used a method of control called ***full state feedback***. As the name implies, this is a control law that generates inputs based on every state of the system. Conversely, partial state feedback is a control law that generates inputs based on only some states of the system. Partial state feedback is a useful technique when the system does not have enough sensors to measure each state, or, like in our case, there are some uncontrollable states. 

We have the state space realization of our system:

\begin{align}
\dot{\tilde{\mathbf{z}}} &= \tilde{A} \tilde{\mathbf{z}} + \tilde{B}\mathbf{u} \\ 
\begin{bmatrix}\dot{\mathbf{z}_\mathcal{C}} \\ \dot{\mathbf{z}_\mathcal{U}} \end{bmatrix} &= \begin{bmatrix}A_\mathcal{C} & A_{12} \\ 0 & A_\mathcal{U}\end{bmatrix} \begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix} + \begin{bmatrix}B_\mathcal{C} \\ 0 \end{bmatrix}\mathbf{u}.
\end{align} 

Suppose for this system we design some control law:
\begin{align}
\mathbf{u} &= -\tilde{K}\tilde{\mathbf{z}}.
\end{align}

We can seperate the control gains into blocks that multiply the controllable states and those that multiply the uncontrollable states:
\begin{align}
\tilde{K} &= \begin{bmatrix} K_\mathcal{C}&K_\mathcal{U} \end{bmatrix} \\
\mathbf{u} &= -\begin{bmatrix} K_\mathcal{C}&K_\mathcal{U} \end{bmatrix}\begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix},
\end{align}
where $K_\mathcal{C}$ is an $n \times m_c$ matrix and $K_\mathcal{U}$ is an $n \times m_u$ matrix. By having both $K_\mathcal{C}$ and $K_\mathcal{U}$ gains, this control law is a full state feedback law.

Investigating the closed-loop response of our realization under this control law:
\begin{align}
\begin{bmatrix}\dot{\mathbf{z}_\mathcal{C}} \\ \dot{\mathbf{z}_\mathcal{U}} \end{bmatrix} &= \begin{bmatrix}A_\mathcal{C} & A_{12} \\ 0 & A_\mathcal{U}\end{bmatrix} \begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix} - \begin{bmatrix}B_\mathcal{C} \\ 0 \end{bmatrix}\begin{bmatrix} K_\mathcal{C}&K_\mathcal{U} \end{bmatrix}\begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix} \\
 &= \begin{bmatrix}A_\mathcal{C}-B_\mathcal{C}K_\mathcal{C} & A_{12}-B_\mathcal{C}K_\mathcal{U} \\ 0 & A_\mathcal{U}\end{bmatrix} \begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix}.\\
\end{align}

Let
\begin{align}
\tilde{F} &= \begin{bmatrix}A_\mathcal{C}-B_\mathcal{C}K_\mathcal{C} & A_{12}-B_\mathcal{C}K_\mathcal{U} \\ 0 & A_\mathcal{U}\end{bmatrix} ,
\end{align}
then
\begin{align}
\begin{bmatrix}\dot{\mathbf{z}_\mathcal{C}} \\ \dot{\mathbf{z}_\mathcal{U}} \end{bmatrix} &= \tilde{F}\begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix}.
\end{align}

Recall that we can characterize the closed-loop response of a system by investigating the eigenvalues of the closed-loop response matrix. Looking at the eigenvalues of $\tilde{F}$:

\begin{align}
\lambda_\tilde{F} &:= \det(\tilde{F} - \lambda_\tilde{F}I) = 0 \\
\det(\tilde{F} - \lambda_\tilde{F}I) &= \begin{vmatrix}(A_\mathcal{C}-B_\mathcal{C}K_\mathcal{C})-\lambda_\tilde{F}I & A_{12}-B_\mathcal{C}K_\mathcal{U} \\ 0 & A_\mathcal{U}-\lambda_\tilde{F}I\end{vmatrix} \\
&= \left|(A_\mathcal{C}-B_\mathcal{C}K_\mathcal{C})-\lambda_\tilde{F}I\right|\cdot\left|A_\mathcal{U}-\lambda_\tilde{F}I\right|
\end{align}

Thus, we find that the eigenvalues of $\tilde{F}$ are the union of the eigenvalues of $A_\mathcal{C}-B_\mathcal{C}K_\mathcal{C}$ and the eigenvalues of $A_\mathcal{U}$. Clearly, the gains multiplying the uncontrolled states, $K_\mathcal{U}$, have no effect on the closed-loop response of the system because they do not appear as arguments in the eigenvalues of the closed-loop response matrix. As such, with no impact of the asymptotic stability of our linear approximation of the system, we can simply set $K_\mathcal{U}=0$, yielding our partial feedback control law:

\begin{align}
\mathbf{u} &= -\begin{bmatrix} K_\mathcal{C}&0 \end{bmatrix}\begin{bmatrix}\mathbf{z}_\mathcal{C} \\ \mathbf{z}_\mathcal{U} \end{bmatrix}.
\end{align}

By design, $(A_\mathcal{C}, B_\mathcal{C})$ is a controllable subsystem. Therefore, we know that we can place the eigenvalues of this subsystem anywhere we like. To ensure the subsystem is stable, we need only select a set of control gains that ensure the real parts of the eigenvalues are all negative. For example, in previous projects, we have discussed how to do with via both ***pole placement*** and solving a ***linear quadratic problem***.

However, even if the eigenvalues of the $(A_\mathcal{C}, B_\mathcal{C})$ subsystem suggest asymptotic stability, for the entire system to be asymptotically stable, we also need the uncontrollable subsystem $(A_\mathcal{U}, 0)$ to be asymptotically stable. Calculating the eigenvalues of $A_\mathcal{U}$, we find:

In [ ]:
s, V = np.linalg.eig(A_u)
print("Eigenvalue(s) of the Uncontrollable Subsystem:")
N(Matrix(s), 3)

This is a tricky situation because the real part of the eigenvalue of $A_\mathcal{U}$ is neither positive (indicating unbounded exponential divergence from $\mathbf{0}$) nor negative (indicating asymptotic exponential convergence to $\mathbf{0}$).

### 4g. Marginal Stability

A mode with an eigenvalue whose real part is $0$ and whose imaginary part is unrestricted, which we will call a $0$ eigenvalue mode, can be classified as either unstable or ***marginally stable*** (also sometimes called ***neutrally stable***). $0$ eigenvalue modes that are unstable diverge from their stationary value in an unbounded, non-accelerating manor when displaced. On the other hand, $0$ eigenvalue modes that are marginally stable neither return to their stationary value, nor diverge from their stationary value when displaced. Instead, they lie somewhere inbetween our common understanding of stability and instability.

For example, consider a mass on a spring with no damping. This system has a $0$ eigenvalue mode, but is it unstable or marginally stable? When displaced from its neutral position, the mass will start to oscillate about its neutral position. These oscillations will neither stop, nor grow in magnitude. Accordingly, the mass will neither converge to its neutral position, nor will it diverge without limit from its neutral position: it is said to be marginally stable.

Consider now a ball in a slingshot, with no friction or gravity. This system is identical to a mass on a spring, but where the spring constant becomes $0$ when the mass leaves the slingshot pouch. Again, this system again has a $0$ eigenvalue mode, but, when pulled back from the neutral position and released, instead of oscillating, the mass now shoots out of the slingshot and maintains a constant velocity. The mass unboundedly diverges from its neutral point *without accelerating*: it is said to be unstable.

As it turns out, there is a method to determine if a $0$ eigenvalue mode of a state space system is marginally stable: a $0$ eigenvalue mode of a state space system is marginally stable if and only if it is a simple root (i.e., its geometric multiplicity is equal to its algebraic multiplicity). The proof of this fact is out of the scope of this project; however, we can still use it to determine the stability of the uncontrollable subsystem, $A_\mathcal{U}$.

We can measure the algebraic multiplicity of the $0$ modes of $A_\mathcal{U}$ by counting the number of eigenvalues of $A_\mathcal{U}$ that are $0$:

In [ ]:
# Calculate the eigenvalues, s, and eigenvectors, V, of A_u
s, V = np.linalg.eig(A_u)

# Count the number of eigenvalues that are 0
algebraic_multiplicity = np.sum(np.isclose(s, 0.0))
print(f'Algebraic multiplicity of 0 modes of A_U: {algebraic_multiplicity}')

We can measure the geometric multiplicity of the $0$ modes of $A_\mathcal{U}$ by counting the number of linearly independent eigenvectors associated with eigenvalues of $0$:

In [ ]:
# Determine which eigenvectors of A_u are associated with 0 eigenvalues
cols = np.isclose(s, 0.0)
eigenvectors = V[:, cols]

# Calculate geometric multiplicity by finding the rank (number of linearly independent) of the associated eigenvectors
geometric_multiplicity = np.linalg.matrix_rank(eigenvectors)
print(f'Geometric multiplicity of 0 modes of A_U: {geometric_multiplicity}')

In this case, we have found that the $0$ modes of $A_\mathcal{U}$ have both algebraic and geometric multiplicity of $1$. This tells us that, indeed, the $0$ modes of $A_\mathcal{U}$ are marginally stable: they will neither diverge to infinity nor converge to 0 when perturbed.

The last step in determining the overall stability of the uncontrollable subsystem is checking the nonzero eigenvalues of $A_\mathcal{U}$. Even though all $0$ modes of $A_\mathcal{U}$ are marginally stable, if there is a nonzero mode that is unstable, i.e., it has an eigenvalue whose real part is greater than $1$, the uncontrollable subsystem would still diverge. In this case, we have already confirmed that $A_\mathcal{U}$ only has one eigenvalue and its real part is $0$. Therefore, we can confirm that no other modes of the uncontrollable subsystem will diverge. As a results, we conclude that, overall, the unstable subsystem is marginally stable.

Were the uncontrollable subsystem unstable, we would have no luck with controlling our spacecraft. No matter what control inputs we applied, the uncontrollable subsystem would always diverge to infinity. However, because the uncontrollable subsystem is marginally stable, and because we know we can force the controllable subsystem to be stable, we know that our system will not diverge unboundedly. Instead, the controllable subsystem will converge to $0$ (the equilibrium point) and the uncontrollable subsystem, which is comprised of mostly the gimbal angles, will stay near some (possibly) nonzero value.

As it turns out, nonlinear analysis can be used to determine the exact behavior of the unstable subsystem. Whereas, conducting this analysis is out of the scope of this project, the results of this analysis on our system indicate that the gimbals will always return to some constant value that is a function of the initial condition of the entire system.

For example, suppose that the initial gimbal angle is $45$ degrees, the controllable subsystem is initially at our selected equilibrium point, and $\theta_{des}=0$. At time $t=2$, suppose $\theta_{des}$ increases to $10$ degerees. The gimbals will turn away from $45$ degrees to increase the pitch rate of the spacecraft until the spacecraft is pointing in the correct direction. When $\theta=\theta_{des}$, the gimbals will return to their initial value of $45$ degrees. Whereas there are no control inputs that we can apply to change the value that the gimbals will settle on, they will always settle on some value dependent on the initial conditions of the entire system. 

Suppose now that we set the initial gimbal angle to our selected $\gamma_e$ but keep the initial state of the rest of the system at equilibrium. Now, after turning to enforce the desired pitch angle, the gimbals will then return to $\gamma_e$. This means that the entire system settles on the selected equilibrium value, which is an ideal behavior.

Summarizing: the unstable subsystem is marginally stable. This means that, as the controllable system settles to an equilibrium value, it will stay around some value. As it turns out, for this system, the uncontrollable subsystem will actually settle on a value, and that value is wholly determined by the initial conditions of the system, i.e., it cannot be altered by control inputs. This is the definition of uncontrollability. Finally, in the case where the controllable subsystem is initially at equilibrium, the value that the uncontrollable subsystem settles on is exactly equal to its initial value.

This behavior tells us that, indeed, if we build a controller to stabilize the controllable subsystem, the uncontrollable subsystem will behavior in a manner that is agreeable.

### 4h. Implementing the Controller

We can now design and implement a controller for our controllable subsystem. In the Pendulum Cart series, we learned how to design and solve a ***Linear Quadratic Problem*** for our state space model to find an optimal controller, called a ***Linear Quadratic Regulator***. Repeating this process now, but instead on our controllable subsystem $(A_\mathcal{C}, B_\mathcal{U})$, we can find a set of optimal control gains, $K_\mathcal{C}$.

Previously, we used ***Bryson's Rule*** to select an untuned ***state cost matrix, $Q$*** and untuned ***input cost matrix, $R$***. We can do this again now, but must remember to transform our state limits into controllable and uncontrollable space via $T$.

In [ ]:
# Define the maximum allowable values for each linear state
#                 omega_theta  theta   omega_gamma  gamma
z_max = np.array([0.1745,      0.0524, 0.2618,      0.7854])

# Transform the maximum allowable linear state to controllable / uncontrollable space
z_tilde_max = T.T @ z_max

# Ignore the uncontrollable part of z_tilde_max
z_tilde_max = z_tilde_max[:-1]

# Use Bryson's rule to get Q
Q_c = np.diag( 1.0 / (z_tilde_max**2) )

In [ ]:
# Define the maximum allowable values for each linear input
#                 tau_gamma
u_max = np.array([0.001])

# Use Bryson's rule to get R
R_c = np.diag( 1.0 / (u_max**2) )

In [ ]:
# Solve the CARE and calculate the gain matrix
P_c = sci.linalg.solve_continuous_are(np.array(A_c, dtype=float), np.array(B_c, dtype=float), Q_c, R_c)
K_c = np.linalg.inv(R_c)@B_c.T@P_c
K_c = np.array(K_c, dtype=float)

# Print the results
print('Control gains for controllable subspace:')
N(Matrix(K_c),3)

We now have an optimal choice of $K_\mathcal{C}$. Recalling all of our definitions up to this point we have:

\begin{align}
\mathbf{u} &= -\begin{bmatrix} K_\mathcal{C} & 0 \end{bmatrix} \tilde{\mathbf{z}}\\
\tilde{\mathbf{z}} &= T^{-1}\mathbf{z} \\
\mathbf{z} &= \mathbf{x}-\mathbf{x}_{des} \\
\mathbf{x} &= \mathbf{m} - \mathbf{m}_e \\
\mathbf{u} &= \mathbf{n} - \mathbf{n}_e \\
\end{align}

Combining these terms, we have the expanded control law: 
$$
\mathbf{n} = \mathbf{n}_e - \begin{bmatrix} K_\mathcal{C} & 0 \end{bmatrix}\left(T^{-1}\left(\mathbf{m} - \mathbf{m}_e - \mathbf{x}_{des}\right)\right)
$$
This is what we will implement in code to build our controller.

Let's do this now. This project's provided simulation function expects us to implement a controller function that takes as arguments
1. The nonlinear state of the system, $\mathbf{m}$, and
2. Some desired pitch angle, $\theta_{des}$.

Given these values, it expects us to return the torque applied to the gimbals, $\tau_\gamma$, as a scalar.

In [ ]:
def controller(state, theta_des):
    """
    The controller function. Given some state information and a target pitch, calculates the torque to apply to the gimbals 
    that (hopefully) points the spacecraft in the desired direction.

    Parameters
    ----------
    state : dictionary of floats with the following keys
        omega_theta : float
            The angular rate of the chassis pitch in radians / second
        omega_gamma : float
            The angular rate of the gimbals in radians / second
        theta : float
            The pitch angle of the chassis in radians
        gamma : float
            The angle of the gimbals in radians
    theta_des : float
        The target pitch angle of the spacecraft in radians

    Returns
    -------
    tau_gamma : float
        The torque to apply to the gimbals
    """
    # Definitions
    m_e = np.array([0, 0, 0, 0.5*np.pi])   # Define the equilibrium nonlinear state vector
    n_e = np.array([0])                    # Define the equilibrium nonlinear input vector
    x_des = np.array([0, theta_des, 0, 0]) # Define a desired linear state vector
    K_c = np.array([[-5.38e-05, -2.02e-02, 5.50e-03]])                # Define the controllable subsystem gain matrix
    K = np.hstack((K_c, np.zeros((len(n_e), len(m_e)-K_c.shape[1])))) # Define the combined gain matrix [K_c, 0]
    T = np.array([[ 0.9995, -0.0000,  0.0000,  0.0302],               # Define the Kalman decomposition transform
                  [ 0.0000,  0.9999,  0.0107, -0.0000],
                  [ 0.0000, -0.0107,  0.9999, -0.0000],
                  [-0.0302, -0.0000,  0.0000,  0.9995]])
    
    # Build the nonlinear state vector
    m = np.array([state['omega_theta'], 
                  state['theta'], 
                  state['omega_gamma'], 
                  state['gamma'],])

    # Build the linear state vector
    x = m - m_e

    # Build the reference tracking linear state vector
    z = x - x_des

    # Transform the reference tracking linear state vector into controllable and uncontrollable subspaces
    z_tilda = T.T @ z

    # Apply the feedback control law with our selected gain matrix to get the linear input vector
    u = -K@z_tilda

    # Convert the linear input vector into the nonlinear input vector
    n = u + n_e

    # Return the nonlinear torque as a scalar
    tau_gamma = n[0]
    return tau_gamma

Now that we have a controller, we can test it in simulation.

## 5. Running a Simulation

The backend of the simulation has already been made for you. It handles building the simulation environment, running and visualizing the simulation, applying your controller's inputs, and tracking relevant data. Let's import this backend now. It is a function named `run` that is stored in a Python script named `opposed_sacmg.py`. 

In [ ]:
# Import the project's backend.
from opposed_sacmg import run

Now we are ready to run the simulation and collect data. To do this, we simply call the `run` function as pass as arguments the initial angle of the gimbals, the desired pitch program number, and the controller function we just built. The initial values of the rest of the nonlinear states are set to 0.

The program number is an integer that selects which of the desired pitch programs will be run. Each program gives a different sequence of desired pitches as a function of time. The valid program numbers are 1-12, inclusively.
* 1-3 are step programs (at t=2 seconds, the desired pitch instantaneously steps up to a new value)
* 4-6 are sequential step programs (every few seconds, the desired pitch instantaneously steps up to a new value)
* 7-9 are linear ramp programs (the desired pitch is a linear function of time)
* 10-12 are sinusoidal programs (the desired pitch follows a sine function)

Once called, the system will be initialized to the selected initial state, a sequence of desired pitches dependent on the program number will be generated, controller will be applied, the nonlinear physics of system will be simulated. When finished, data collected during simulation will be returned. The returned data is a dictionary with the values:

`data['time']` : list of n floats

    The time, in seconds, at which each data point is collected
    
`data['omega_theta']` : list of n floats

    The pitching rate of the spacecraft, in radians per second, at each of the n data collection points.

`data['theta']` : list of n floats

    The pitch angle of the spacecraft, in radians, at each of the n data collection points.

`data['omega_gamma']` : list of n floats

    The angular rate of the gimbals, in radians per second, at each of the n data collection points.
    
`data['gamma']`: list of n floats

    The angle of the gimbals, in radians, at each of the n data collection points.

`data['tau_gamma']` : list of n floats

    The torque applied to the gimbals, in Newton-meters, at each of the n data collection points.

`data['theta_des']` : list of n floats

    The desired pitch angle of the spacecraft, in radians, at each of the n data collection points.

Let's run a simulation and test our controller!

In [ ]:
# Run the simulation with pitch program 1 and collect the simulation data
data = run(0.5*np.pi, 1, controller)

Voila! Again our controller successfully drives the system to the desired pitch and stays there. We can confirm the by plotting the results:

In [ ]:
# Import plotting tool
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Plot the pitch, desired pitch, and input torque as functions of time
fig, ax = plt.subplots(1)
ax.plot(data['time'], data['theta']*180/3.14, label='Pitch [deg]', lw=2.0, c='r')
ax.plot(data['time'], data['theta_des']*180/3.14, label='Desired Pitch [deg]', lw=2.0, c='r', ls='--')
ax.plot(data['time'], 1000*data['tau_gamma'], label='Torque [mN-m]', lw=2.0, c='b')
ax.legend()
ax.set_xlabel('Time [seconds]')
ax.axhline(c='k', lw=0.5)
plt.show()

Further, just as expected, once the rest of the system settled, the gimbals returned to their initial angles. We can confirm this by plotting the gimbal angles as a function of time:

In [ ]:
# Plot the gimbal angles as a function of time
fig, ax = plt.subplots(1)
ax.plot(data['time'], 180.0*data['gamma']/np.pi, label='Gimbal Angle [deg]', lw=2.0, c='k')
ax.axhline(180.0*data['gamma'][0]/np.pi, c='k', lw=2.0, ls='--', label='Initial Gimbal Angle [deg]')
ax.legend()
ax.set_xlabel('Time [seconds]')
plt.show()

Through much analysis and simulation, we have proved that, even though there is a marginally stable uncontrollable subsystem, we can still build a controller that points the spacecraft to desired pitch angle.

## Assignment